# SPCM3-only alignment — interactive analysis

For runs of `experiments/atom_align_spcm3_only.py` (`*-AtomAlignSPCM3Only.h5`), which records **SPCM3 alone** — no SPCM1/2 reference.

- **A** — histogram of every SPCM3 window, with the run's `readout_threshold` and the threshold found from the data (Otsu)
- **B** — the same counts as a time series, so individual atom events are visible

**Watch `separation`, not `contrast`.** With no independent reference, the experiment's `align_contrast` splits SPCM3 against `readout_threshold` — so signal windows are above it by construction and background below it. That number cannot drop below roughly the threshold spacing, and it moves when you change the threshold even if the optics never moved. This notebook instead finds the split from the data and reports `separation = (mean_above − mean_below) / combined σ`, which does not depend on what you dialled in. **`separation > 2` = two genuinely resolved peaks.** Near 0 = one blob, SPCM3 is not seeing the atom.

It also prints a **suggested threshold** — set `readout_threshold` on the dashboard to about that value.

All logic is imported from `analyze_align_spcm3_only.py`; this notebook keeps no copy of it.

In [ ]:
# ---------------------------------------------------------------------------
# KERNEL CHECK -- run this FIRST.
# matplotlib cannot draw in the `artiq7_claude` env (numpy 1.23.5, broken BLAS:
# its affine transforms call np.dot while rendering, so plt.hist SEGFAULTS and
# the client reports "The Kernel crashed"). This analysis needs only
# numpy / h5py / matplotlib, NOT artiq -- so run it on plain Python 3.11:
#     VS Code -> kernel picker (top right) -> Python 3.11
# The probe runs np.dot in a SUBPROCESS, so a broken BLAS kills the child
# process instead of this kernel.
# ---------------------------------------------------------------------------
import subprocess, sys

_blas = subprocess.run(
    [sys.executable, "-c",
     "import numpy as np; np.dot(np.ones((8, 8)), np.ones((8, 8)))"],
    capture_output=True)
if _blas.returncode != 0:
    print("!" * 74)
    print("!! BROKEN BLAS in this kernel -- plotting WILL crash it.")
    print("!!   kernel : {}".format(sys.executable))
    print("!!   fix    : switch the kernel to Python 3.11, then re-run.")
    print("!" * 74)
else:
    print("BLAS ok -- plotting is safe in this kernel.")
    print("kernel:", sys.executable)

# Backend by what is importable -- never a bare `%matplotlib widget` (it needs
# ipympl; without it the kernel falls back to Qt, where plt.show() blocks).
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
except ImportError:
    get_ipython().run_line_magic("matplotlib", "inline")

import os, importlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

sys.path.insert(0, os.getcwd())
import analyze_align_spcm3_only as A
importlib.reload(A)          # picks up edits to the .py without restarting
from analyze_align_spcm3_only import (find_run, find_runs, load_run,
                                      summarize, analyze, otsu_threshold)

print("matplotlib backend:", matplotlib.get_backend())
print("runs on disk:", len(find_runs()))

## One run

`RID = None` uses the latest run. Prints the statistics, shows the figure, and saves a PNG next to the `.h5`.

In [ ]:
RID = None   # None = latest run; or e.g. RID = 2452

fig = analyze(find_run(RID))

In [ ]:
# The 10 most recent runs on disk:
for p in find_runs()[-10:]:
    print(A._rid_of(p), p)

## While you walk the detector

Re-run this cell after each adjustment (take a short run, then execute). `separation` is the number to maximize; `suggested` is what to set `readout_threshold` to on the dashboard.

In [ ]:
res = summarize(find_run(RID), verbose=False)
if res is None:
    print("no windows in that run")
else:
    s = res["otsu_split"]
    if s is None:
        print("RID %s: all %d windows have the same count (%.0f) -- nothing to split"
              % (res["rid"], res["n_windows"], res["mean"]))
    else:
        print("RID %s | %d windows | mean %.1f counts"
              % (res["rid"], res["n_windows"], res["mean"]))
        print("separation = %.2f sigma   (>2 = two resolved peaks)" % s["separation"])
        print("suggested readout_threshold = %.0f" % s["threshold"])
        print("peaks: %.1f (n=%d)  vs  %.1f (n=%d)"
              % (s["mean_above"], s["n_above"], s["mean_below"], s["n_below"]))

## Raw counts — your own binning

`load_run` returns `(c3, meta, rid)`: `c3` is one count per imaging window; `meta` carries `readout_threshold`, `gate` [s], `n_windows`, and `source` (whether you got the full record `spcm_all3` or only the rolling `spcm_counts3`).

In [ ]:
c3, meta, rid = load_run(find_run(RID))
print("RID", rid, "|", meta["source"], "|", meta["n_windows"], "windows",
      "| gate %.0f ms" % (1e3 * meta["gate"]))

if c3.size and c3.max() > c3.min():
    bins = np.arange(c3.min(), c3.max() + 2) - 0.5   # one bin per integer count
    plt.figure(figsize=(7, 4))
    plt.hist(c3, bins=bins, color="C0", alpha=0.85)
    t = otsu_threshold(c3)
    if t is not None:
        plt.axvline(t, color="C2", lw=1.4, label="suggested = %d" % t)
        plt.legend()
    plt.xlabel("SPCM3 counts per window")
    plt.ylabel("windows")
    plt.tight_layout()
else:
    print("nothing to histogram (no windows, or every count identical)")